# Feature Reduction Ablation Study — Accuracy Impact

This notebook summarises how **greedy correlation-based pruning** affects end-to-end pipeline accuracy.
Three pruning thresholds are compared against the full-feature baseline.

---

## Experiment Configurations

| Config | Manual feats | TF-IDF feats | Total | Threshold |
|:---|:---:|:---:|:---:|:---:|
| **A — Baseline (all features)** | 23 | 27 | **50** | — |
| **B — No TF-IDF** | 23 | 0 | **23** | — |
| **C — Pruned \|r\| ≥ 0.85** | 13 | 8 | **21** | 0.85 |
| **D — Pruned \|r\| ≥ 0.75** | 19 | 6 | **25** | 0.75 |

### Features dropped / kept per config

**Config C  (|r| ≥ 0.85)**
- Manual dropped (3): `approver`, `conversion rate_is_null`, `payment authorizer`
- TF-IDF kept (8): `card`, `cypress hill`, `financial`, `platform`, `purchase`, `renewal`, `spring`, `use`

**Config D  (|r| ≥ 0.75)**
- Manual dropped (4): `approver`, `conversion rate_is_null`, `transaction type_purchase`, `transaction type_refund`
- TF-IDF kept (6): `card`, `cypress hill`, `financial`, `purchase`, `spring`, `use`

---

## Overall Results

| Metric | A — Baseline (50) | B — No TF-IDF (23) | C — |r|≥0.85 (21) | D — |r|≥0.75 (25) |
|:---|:---:|:---:|:---:|:---:|
| **Test Accuracy** | **87.69 %** | 83.67 % | 80.40 % | 65.58 % |
| **F1 (weighted)** | **85.6 %** | 82.0 % | 79.4 % | 64.2 % |
| **F1 (macro)** | **78.2 %** | 74.3 % | 64.0 % | 58.1 % |
| **ML-only accuracy** | **84.16 %** | 78.88 % | 74.59 % | 55.12 % |
| **Deterministic accuracy** | 98.95 % | 98.95 % | 98.95 % | 98.95 % |

---

## Per-Class F1 Comparison (key classes)

| GL Code | A — Baseline | B — No TF-IDF | C — |r|≥0.85 | D — |r|≥0.75 |
|:---|:---:|:---:|:---:|:---:|
| 5700 office expense | 0.89 | 0.84 | 0.79 | 0.63 |
| 5773 software license | 0.90 | 0.85 | 0.78 | 0.58 |
| 5450 hr recruiting | 0.97 | 0.96 | 0.95 | 0.87 |
| account payable sage | 0.85 | 0.81 | 0.86 | 0.43 |
| 5784 travel entertainment | 0.67 | 0.52 | 0.54 | 0.36 |
| 5785 meal entertainment | 0.35 | 0.30 | 0.22 | 0.18 |

---

## Key Takeaways

1. **Deterministic rules are unaffected** — all four configs score 98.95 % on the 95 rule-matched samples.
2. **TF-IDF features matter significantly**: removing them entirely (Config B) costs ~4 pp accuracy; aggressive pruning at |r|≥0.75 (Config D) costs **22 pp** because only 6 of 27 survive.
3. **The 0.85 threshold (Config C) is far more reasonable than 0.75** — it preserves 8 TF-IDF features and 13 manual features (21 total), achieving **80.40 % accuracy** versus 65.58 % for the 0.75 cutoff.
4. **Still a ~7 pp gap vs baseline**: even at 0.85, the pruned set loses meaningful signal. The biggest drops are in the two largest classes: *5700 office expense* (−10 pp F1) and *5773 software license* (−12 pp F1).
5. **Recommendation**: correlation pruning at these thresholds is too aggressive for this dataset. A lighter approach — e.g., reducing `max_features` to 15–18 in the TF-IDF vectoriser, or raising the threshold to |r| ≥ 0.95 (which only drops duplicate-level pairs) — would give a better accuracy / dimensionality trade-off.